In [13]:
from unike.module.model import RGCN, CompGCN
import sys
import pandas as pd

sys.path.extend(['.', '..'])

from q import link, drug_ent_indexs, add_id

In [14]:
RGCN_model = RGCN(
	ent_tol = 121649,
	rel_tol = 22,
	dim = 200,
	num_layers = 2
)
RGCN_model.load_checkpoint("/home/wangtao/src/kg4rd/src/kg4rd/kge/checkpoints/A/RGCN_entrie_Accel_20250910-1000.pth")

CompGCN_model = CompGCN(
    ent_tol = 121649,
    rel_tol = 22,
    dim = 50
)
CompGCN_model.load_checkpoint("/home/wangtao/src/kg4rd/src/kg4rd/kge/checkpoints/A/CompGCN_entrie_Accel_20250910-1000.pth")

In [15]:
RGCN_result = add_id(link.link(
    drug_ent_indexs,
    [1],
    [9315, 4289, 13239, 7993, 9016, 7509], # UTRN, COL6A3, DYSF, DOK7, DMD, SGCD
    RGCN_model, 'cuda:0'
))

In [16]:
CompGCN_result = add_id(link.link(
    drug_ent_indexs,
    [1],
    [9315, 4289, 13239, 7993, 9016, 7509], # UTRN, COL6A3, DYSF, DOK7, DMD, SGCD
    CompGCN_model, 'cuda:0'
))

In [17]:
topk = 10000

RGCN_top = RGCN_result.head(topk)
CompGCN_top = CompGCN_result.head(topk)

RGCN_pairs = set(zip(RGCN_top['head'], RGCN_top['tail']))
CompGCN_pairs = set(zip(CompGCN_top['head'], CompGCN_top['tail']))
intersect_pairs = RGCN_pairs & CompGCN_pairs

mask_rgcn = list(zip(RGCN_top['head'], RGCN_top['tail']))
mask_comp = list(zip(CompGCN_top['head'], CompGCN_top['tail']))
rgcn_inter = RGCN_top[[p in intersect_pairs for p in mask_rgcn]]
comp_inter = CompGCN_top[[p in intersect_pairs for p in mask_comp]]
# 同一 (head, rel, tail)，两模型分数等列用后缀区分
intersect_merged = rgcn_inter.merge(
    comp_inter, on=['head', 'rel', 'tail'], how='inner', suffixes=('_RGCN', '_CompGCN')
)

# 标准 CSV 不支持多 sheet，用 xlsx
out_xlsx = f"./target_models_top{topk}.xlsx"
with pd.ExcelWriter(out_xlsx, engine='openpyxl') as writer:
    RGCN_top.to_excel(writer, sheet_name='RGCN', index=False)
    CompGCN_top.to_excel(writer, sheet_name='CompGCN', index=False)
    intersect_merged.to_excel(writer, sheet_name='intersect', index=False)